In [ ]:
# ============================================================
# [목적]
#   CPU/Memory 분포가 롱테일 -> 로그 변환으로 꼬리 구간 학습 개선
#   피처 변환: log(cpu + 1e-6), log(memory + 1e-6)
#   타겟 변환: log(energy_kwh) -> 예측 후 exp() 역변환
#   duration, hour: 변환 없음
#   train만 증강 (3600초 cap)
# ============================================================

In [1]:
import os
import yaml
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import gc
import json

In [2]:
CONFIG_PATH = "/content/config.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT     = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'])
MODEL_PATH     = os.path.join(DRIVE_ROOT, config['paths']['models'])
REPORT_PATH    = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports")

hourly_file = os.path.join(PROCESSED_PATH, "instance_usage_hourly_processed.parquet")
df = pd.read_parquet(hourly_file)
print(f"Hourly rows: {len(df):,}")

Hourly rows: 1,454,606


In [3]:
# ============================================================
# 로그 변환
# ============================================================
EPSILON = 1e-6

df['cpu_log']  = np.log(df['cpu'] + EPSILON)
df['memory_log'] = np.log(df['memory'] + EPSILON)
df['energy_log'] = np.log(df['energy_kwh'])

print(f"\n[로그 변환 전 cpu 분포]")
print(df['cpu'].describe())
print(f"\n[로그 변환 후 cpu_log 분포]")
print(df['cpu_log'].describe())


[로그 변환 전 cpu 분포]
count    1.454606e+06
mean     7.017573e-03
std      7.709754e-03
min      0.000000e+00
25%      2.771552e-03
50%      4.874045e-03
75%      8.921941e-03
max      3.457098e-01
Name: cpu, dtype: float64

[로그 변환 후 cpu_log 분포]
count    1.454606e+06
mean    -5.511102e+00
std      1.283668e+00
min     -1.381551e+01
25%     -5.887987e+00
50%     -5.323626e+00
75%     -4.719130e+00
max     -1.062153e+00
Name: cpu_log, dtype: float64


In [4]:
# ============================================================
# 피처 / 타겟
# ============================================================
features = ["cpu_log", "memory_log", "duration", "hour"]
target = "energy_log"

X = df[features]
y = df[target]

# ============================================================
# Train / Test 분할 (test는 원본 그대로!)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)

# 역변환 평가용 원본 energy_kwh
_, _, _, y_test_orig = train_test_split(
    X, df['energy_kwh'],
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)

print(f"\nTrain (원본): {len(X_train):,}")
print(f"Test  (원본): {len(X_test):,}")


Train (원본): 1,163,684
Test  (원본): 290,922


In [5]:
# ============================================================
# Train만 증강 (duration x[2,3,4,6,8,12], 3600초 cap)
# ============================================================
MAX_DURATION = 3600.0

augmented = [pd.concat([X_train, y_train.rename("energy_log")], axis=1)]

for multiplier in [2, 3, 4, 6, 8, 12]:
    df_aug = pd.concat([X_train, y_train.rename("energy_log")], axis=1).copy()

    new_duration = (df_aug['duration'] * multiplier).clip(upper=MAX_DURATION)
    actual_mult  = new_duration / df_aug['duration']

    df_aug['duration']   = new_duration
    # log(energy * mult) = log(energy) + log(mult) -> log 공간에서 덧셈
    df_aug['energy_log'] = df_aug['energy_log'] + np.log(actual_mult)

    augmented.append(df_aug)

df_train_aug = pd.concat(augmented, ignore_index=True)
del augmented; gc.collect()

X_train_aug = df_train_aug[features]
y_train_aug = df_train_aug[target]
del df_train_aug; gc.collect()

print(f"Train (증강): {len(X_train_aug):,}")

Train (증강): 8,145,788


In [6]:
# ============================================================
# RF 학습
# ============================================================
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=config['model']['random_state'],
    n_jobs=-1
)
rf.fit(X_train_aug, y_train_aug)
print(f"\nRF trained!")
print(f"feature_names_in_: {rf.feature_names_in_}")



RF trained!
feature_names_in_: ['cpu_log' 'memory_log' 'duration' 'hour']


In [8]:
# ============================================================
# 평가 (log 공간 예측 -> exp() 역변환)
# ============================================================
y_pred_log = rf.predict(X_test)
y_pred     = np.exp(y_pred_log)       # 역변환
y_true     = y_test_orig.values       # 원본 energy_kwh

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"\n[RF (hourly + log 변환) 성능]")
print(f"  RMSE : {rmse:.8f} kWh")
print(f"  MAE  : {mae:.8f} kWh")
print(f"  R2   : {r2:.8f}")
print(f"  MAPE : {mape:.4f}%")

# Feature Importance
importance = pd.DataFrame({
    'feature'   : features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\n[Feature Importance]")
print(importance.to_string(index=False))


[RF (hourly + log 변환) 성능]
  RMSE : 0.00048730 kWh
  MAE  : 0.00019941 kWh
  R2   : 0.99997685
  MAPE : 0.1451%

[Feature Importance]
   feature   importance
  duration 9.997706e-01
   cpu_log 2.234526e-04
memory_log 5.527114e-06
      hour 4.073084e-07


In [9]:
# ============================================================
# 모델 저장
# ============================================================
model_file = os.path.join(MODEL_PATH, config['model']['model_names']['rf_hourly'])
with open(model_file, 'wb') as f:
    pickle.dump(rf, f)
print(f"\nSaved: {model_file} ({os.path.getsize(model_file)/(1024**2):.1f} MB)")
print(f"feature_names_in_: {rf.feature_names_in_}")

# ============================================================
# 결과 저장
# ============================================================
result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json_hourly'])

results = {
    'method'   : 'RandomForest (hourly + log transform + train augmentation)',
    'features' : features,
    'target'   : target,
    'log_transform': {
        'cpu'       : 'log(cpu + 1e-6)',
        'memory'    : 'log(memory + 1e-6)',
        'energy_kwh': 'log(energy_kwh) -> exp() 역변환',
    },
    'rf_params': {
        'n_estimators'     : 100,
        'max_depth'        : 15,
        'min_samples_split': 10,
        'min_samples_leaf' : 5,
    },
    'augmentation': {
        'multipliers' : [2, 3, 4, 6, 8, 12],
        'max_duration': MAX_DURATION,
        'train_only'  : True,
    },
    'metrics': {
        'rmse': float(rmse),
        'mae' : float(mae),
        'r2'  : float(r2),
        'mape': float(mape),
    },
    'feature_importance': importance.set_index('feature')['importance'].to_dict(),
    'model_file': config['model']['model_names']['rf_hourly'],
}

with open(result_file, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Results saved: {result_file}")
print("Done!")


Saved: /content/drive/MyDrive/EcoTracing/models/energy_model_rf_hourly.pkl (310.1 MB)
feature_names_in_: ['cpu_log' 'memory_log' 'duration' 'hour']
Results saved: /content/drive/MyDrive/EcoTracing/outputs/reports/phase1_results_hourly.json
Done!
